In [1]:
# make models iside src importable here. For uv this does not matter,
# or when using .venv, but does matter when using global
from util import make_dir_modules_importable
make_dir_modules_importable()

Adding: D:\Documents\Ciencias de datos\Ciencia de datos 2\PRACTICAS\uasd-ds-2\unidad01\LAB02\src to sys.path


In [4]:
# fetch data
from lab02.data import fetch_bank_note_data_from_file
backnote_auth_df = fetch_bank_note_data_from_file()

D:\Documents\Ciencias de datos\Ciencia de datos 2\PRACTICAS\uasd-ds-2\unidad01\LAB02


In [15]:
import pandas as pd

columns = list(backnote_auth_df.columns)

print("Audit of the data as it comes from repo: https://archive.ics.uci.edu/dataset/267/banknote+authentication ")
print(f"  - Columns: {columns}")
print("  - Shape: ", backnote_auth_df.shape)
print(f"  - Duplicados: {backnote_auth_df.duplicated().sum()}")

audit = pd.DataFrame({
    "tipo": backnote_auth_df.dtypes.astype(str),
    "ausentes": backnote_auth_df.isna().sum(),
    "porcentaje_ausentes": (backnote_auth_df.isna().mean()*100),
    "unicos": backnote_auth_df.nunique(dropna=False).sum()
})
audit.head(100)

Audit of the data as it comes from repo: https://archive.ics.uci.edu/dataset/267/banknote+authentication 
  - Columns: ['variance', 'skewness', 'curtosis', 'entropy', 'target']
  - Shape:  (1372, 5)
  - Duplicados: 24


,tipo,ausentes,porcentaje_ausentes,unicos
variance,float64,0,0.0,5022
skewness,float64,0,0.0,5022
curtosis,float64,0,0.0,5022
entropy,float64,0,0.0,5022
target,int64,0,0.0,5022


In [17]:
#dropping duplicates
backnote_auth_df = backnote_auth_df.drop_duplicates()

TARGET = "target"
X = backnote_auth_df.drop(columns=[TARGET])
y = backnote_auth_df[TARGET].to_frame()

feature_columns = list(X.columns)
target_columns = list(y.columns)
print("Audit after removing duplicates")
print(f"  - Feature columns: {feature_columns}")
print(f"  - Target columns: {target_columns}")
print("  - Feature: ", X.shape)
print("  - Target: ", y.shape)
duplicate_count = X.duplicated().sum()
print(f"  - Target dentro de X: {list(y.columns)[0] in X.columns}")
print(f"  - Duplicados: {duplicate_count}")
audit.head(100)

Audit after removing duplicates
  - Feature columns: ['variance', 'skewness', 'curtosis', 'entropy']
  - Target columns: ['target']
  - Feature:  (1348, 4)
  - Target:  (1348, 1)
  - Target dentro de X: False
  - Duplicados: 0


,tipo,ausentes,porcentaje_ausentes,unicos
variance,float64,0,0.0,5022
skewness,float64,0,0.0,5022
curtosis,float64,0,0.0,5022
entropy,float64,0,0.0,5022
target,int64,0,0.0,5022


In [21]:
"""
this one does not apply for our dataset, but we will added here for documentation purpose. 
Since our dataset does not include null or categories other than numbers,
this will add nothing to our X values
"""
# 8. Separar tipos y crear preprocesamiento

from sklearn.compose import ColumnTransformer # to applyh transformers to columns
from sklearn.impute import SimpleImputer # Fill NaN values with the median, mean, or mode
from sklearn.pipeline import Pipeline # To chain steps
from sklearn.preprocessing import (
        OneHotEncoder, # to transform categories into numeric, ["Lima", "Quito", "Bogotá"] -> ciudad_Lima, ciudad_Quito, ciudad_Bogotá
        StandardScaler # to put units into equivaletns cuantities, SVM is sensible to units
    )

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

print(len(num_cols), len(cat_cols))

4 0


In [ ]:
# 9. Dividir, baseline y SVM
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, random_state=42, stratify=y)
dummy = Pipeline([
    ("prep", preprocess),
    ("model", DummyClassifier(strategy="most_frequent"))
])

svm = Pipeline([
    ("prep", preprocess),
    ("model", SVC(C= 1, gamma= "scale", probability= True, random_state= 42))
])

for name, model in { "dummy": dummy, "svm": svm }.items():
    model.fit(X_train, y_train)
    pred= model.predict(X_test)
    print(name, f1_score(y_test, pred, average = "macro"))

print(classification_report(y_test, svm.predict(X_test)))

dummy 0.35406698564593303
svm 0.9962642684192321
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       148
           1       0.99      1.00      1.00       122

    accuracy                           1.00       270
   macro avg       1.00      1.00      1.00       270
weighted avg       1.00      1.00      1.00       270



d:\Documents\Ciencias de datos\Ciencia de datos 2\PRACTICAS\uasd-ds-2\unidad01\LAB02\.venv\Lib\site-packages\sklearn\utils\validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Documents\Ciencias de datos\Ciencia de datos 2\PRACTICAS\uasd-ds-2\unidad01\LAB02\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
